In [1]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


项目根目录: /Users/lizhipeng/个人成长/职业发展/求职面试/项目/RetailInsight


# 03 · K-Means 客户聚类
> 目标：回答「会员除了 RFM 规则分群，还能不能被无监督地分成结构不同的群体」。
> 关键结论见文末：**轮廓系数达标与否取决于特征空间，不取决于调参**。

In [2]:
import pandas as pd
from retailinsight.io.loader import read_items, read_parquet
from retailinsight.settings import INTERIM_DIR, ITEMS_DIR, MART_CLUSTER, MART_RFM

rfm = read_parquet(MART_RFM)
txn = read_parquet(INTERIM_DIR / "transactions.parquet")
items = read_items(ITEMS_DIR)
products = read_parquet(INTERIM_DIR / "products.parquet")


## 3.1 候选特征空间
> `base6` = 计划 §8.1 规定的 6 维；`rfm3` = 只含对人群有区分度的核心维度；
> `rfm3_behavior` = 再叠加行为维度（周末占比 / 折扣敏感度 / 季节性强弱）。

In [3]:
from retailinsight.features.cluster_features import (CLUSTER_FEATURE_SPACES, ALL_FEATURES,
                                                     build_cluster_features)
from retailinsight.models.cluster import build_cluster_matrix, choose_k

feats = build_cluster_features(rfm, txn, items, products)
print(f"参与聚类的会员: {len(feats):,}（mart_rfm 共 {len(rfm):,}）")
for name, cols in CLUSTER_FEATURE_SPACES.items():
    print(f"  {name:16s} -> {cols}")


参与聚类的会员: 35,650（mart_rfm 共 35,650）
  base6            -> ['recency_days', 'frequency', 'monetary', 'avg_basket', 'n_categories', 'items_per_txn']
  rfm3             -> ['recency_days', 'frequency', 'monetary']
  rfm3_behavior    -> ['recency_days', 'frequency', 'monetary', 'weekend_ratio', 'discount_sensitivity', 'seasonality_strength']
  rfm3_behavior_p1 -> ['recency_days', 'frequency', 'monetary', 'weekend_ratio', 'discount_sensitivity', 'seasonality_strength', 'promo_response_rate', 'coupon_used_rate', 'store_concentration', 'channel_preference', 'repurchase_cycle_days', 'cross_category_ratio', 'silence_ratio']


## 3.2 逐 K 评估：肘部法 + 轮廓系数
> 轮廓系数在 3.5 万样本上是 O(n²)，因此按固定种子抽样 5000 条计算。

In [4]:
X, _ = build_cluster_matrix(feats, ["recency_days", "frequency", "monetary"])
metrics = choose_k(X)
pd.DataFrame(metrics).T.assign(
    min_share=lambda d: d["cluster_sizes"].apply(lambda s: min(s.values()) / sum(s.values()))
).round(4)


,inertia,silhouette,cluster_sizes,min_share
2,52195.540567,0.480311,"{1: 19715, 0: 15935}",0.4470
3,31829.975905,0.503668,"{1: 17950, 2: 12307, 0: 5393}",0.1513
4,25130.092424,0.416451,"{2: 10722, 0: 10418, 1: 9541, 3: 4969}",0.1394
5,19492.276532,0.423465,"{1: 10707, 3: 10237, 2: 9599, 0: 4976, 4: 131}",0.0037
6,15592.236952,0.443964,"{0: 11030, 5: 9218, 1: 8932, 2: 4028, 3: 2311,...",0.0037
7,12760.261499,0.431034,"{4: 10104, 0: 9214, 2: 8143, 1: 3412, 3: 2427,...",0.0037
8,10415.835463,0.40313,"{1: 8838, 7: 6516, 0: 6419, 4: 6011, 2: 3291, ...",0.0037


## 3.3 神谕上界检验：0.35 到底可达不可达？
> 直接用生成器埋的**真实 8 类人群**当标签算轮廓系数。
> 若神谕值也很低，说明该特征空间本身无法分离真实结构，再调参也无济于事。

In [5]:
from retailinsight.models.cluster import oracle_silhouette

members = read_parquet(INTERIM_DIR / "members.parquet")
persona = members.set_index("member_id")["_persona"]
p = feats["member_id"].map(persona).values

for name, cols in CLUSTER_FEATURE_SPACES.items():
    Xs, _ = build_cluster_matrix(feats, cols)
    print(f"  {name:16s} 神谕上界 silhouette = {oracle_silhouette(Xs, p):.4f}")


  base6            神谕上界 silhouette = 0.0693
  rfm3             神谕上界 silhouette = -0.0563


  rfm3_behavior    神谕上界 silhouette = -0.0389
  rfm3_behavior_p1 神谕上界 silhouette = 0.0125


## 3.4 各特征空间的完整评估与择优

In [6]:
from retailinsight.models.cluster import evaluate_space, select_space

evals = [evaluate_space(feats, name, cols, persona) for name, cols in CLUSTER_FEATURE_SPACES.items()]
for e in evals:
    print(f"  {e['space']:16s} K={e['k']}  silhouette={e['silhouette']:.4f}  "
          f"ARI={e['ari']:.4f}  达标={e['reached_target']}  神谕={e['oracle_silhouette']:.4f}")

choice = select_space(evals, has_persona=True)
print("\n选中:", choice["selected_space"], "（择优准则:", choice["criterion"], "）")


  base6            K=2  silhouette=0.3923  ARI=0.1163  达标=True  神谕=0.0693
  rfm3             K=4  silhouette=0.4165  ARI=0.1339  达标=True  神谕=-0.0563
  rfm3_behavior    K=2  silhouette=0.3993  ARI=0.1104  达标=True  神谕=-0.0389
  rfm3_behavior_p1 K=2  silhouette=0.3636  ARI=0.1019  达标=True  神谕=0.0125

选中: rfm3 （择优准则: ari_vs_persona_then_silhouette ）


## 3.5 簇画像与业务命名

In [7]:
chosen = next(e for e in evals if e["space"] == choice["selected_space"])
chosen["desc"][["cluster_id", "cluster_name", "member_cnt", "member_share", "gmv_share",
                 "active_pct", "frequency_pct", "monetary_pct"]].round(3)


,cluster_id,cluster_name,member_cnt,member_share,gmv_share,active_pct,frequency_pct,monetary_pct
0,0,一般维持,10418,0.292,0.23,0.529,0.545,0.550
1,1,低频低额,9541,0.268,0.04,0.544,0.206,0.202
2,2,高价值活跃,10722,0.301,0.70,0.630,0.847,0.841
3,3,沉睡流失,4969,0.139,0.03,0.073,0.222,0.229


## 3.6 与先验人群的列联表（埋点闭环）
> 计划 §4.2 预埋的 8 类人群在这里终于有了消费方。

In [8]:
cluster = read_parquet(MART_CLUSTER)
ct = pd.crosstab(cluster["cluster_name"],
                 cluster["member_id"].map(persona))
ct


member_id,一次性,促销敏感,周末囤货,季节性,新客尝鲜,沉默流失,高频低额,高频高值
cluster_name,,,,,,,,
一般维持,3,2147,2129,1699,725,2010,910,795
低频低额,1960,765,675,519,2770,2189,367,296
沉睡流失,3200,31,27,2,0,1685,16,8
高价值活跃,0,2416,1531,694,0,640,3003,2438
